# 임장 음성메모 PoC — ③ 오픈소스 LLM으로 항목 추출 (규칙 기반과 비교)

받아쓴 글에서 항목표를 뽑는 일을 **Qwen2.5-7B-Instruct**(공개 LLM, 한국어 가능)로 해 본다.
유료 API 없이 Colab 무료 T4 GPU에서 4비트로 줄여(양자화) 실행한다.

- 입력: 대본 원문 / turbo+소음+힌트 / small+소음+힌트없음 받아쓰기 (각 5건)
- 지시문: Claude 추출기와 **같은 지시문**(`imjang/extract_llm.py`의 SYSTEM)
- 출력: 항목표 JSON → 저장소 `data/llm_extracted/` → `python run.py --method qwen` 으로 채점

**실행 전**: `런타임 → 런타임 유형 변경 → T4 GPU` 후 위에서부터 ▶ (모델 내려받기 포함 약 15분)

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes
import torch; print("GPU:", torch.cuda.is_available())

In [ ]:
SYSTEM = "너는 부동산 임장(현장 방문) 음성메모를 정리하는 도우미다.\n받아쓴 글에서 주어진 항목표에 맞춰 값을 뽑아라.\n\n규칙\n- 말에 실제로 나온 값만 적는다. 나오지 않은 항목은 반드시 null (빈 목록은 []). 추측 금지.\n- 금액은 '만원' 단위 정수: 7억 2천 → 72000, 18만원 → 18, 5천만원 → 5000.\n- 관리비(mgmt_fee)는 만원 단위 숫자.\n- 말하는 사람이 값을 고쳐 말하면(\"아 아니다 ~\") 마지막 값을 쓰고, 그 사실을 uncertain 에 적는다.\n- '~쯤', '~정도라던데', '모르겠다'처럼 불확실한 말은 값을 적되 uncertain 에도 적는다.\n  단, 숫자를 전혀 말하지 않았으면 null.\n- sido 는 '서울특별시' / '인천광역시' / '경기도' 중 하나로 적는다(말에 시·군·구만 나와도 추론 가능).\n- sigungu 는 '부천시 원미구', '성남시 분당구', '영등포구'처럼 적는다.\n- chopuma(초품아): '초품아' 또는 길을 건너지 않고 단지 안/바로 옆 초등학교면 true, 길을 건너야 하면 false, 언급 없으면 null.\n- seller_loan_exists: 매도인 융자(근저당)가 있다고 하면 true, 없다고 하면 false, 언급 없으면 null.\n- defects: 누수·균열·곰팡이·파손·수리 필요 등 하자를 짧은 문장으로. 없으면 [].\n- 받아쓰기 오류로 보이는 단어는 문맥상 명백할 때만 바로잡는다.\n\n출력 형식: 아래 키를 모두 가진 JSON 객체 하나만 출력한다. 설명·코드블록 없이 JSON만.\n- \"sido\": string 또는 null\n- \"sigungu\": string 또는 null\n- \"dong\": string 또는 null\n- \"property_type\": string 또는 null\n- \"sale_price\": integer 또는 null\n- \"jeonse_price\": integer 또는 null\n- \"wolse_deposit\": integer 또는 null\n- \"wolse_monthly\": integer 또는 null\n- \"seller_loan_exists\": boolean 또는 null\n- \"seller_loan_amount\": integer 또는 null\n- \"area_pyeong\": number 또는 null\n- \"built_year\": integer 또는 null\n- \"floor\": string 또는 null\n- \"facing\": string 또는 null\n- \"households\": integer 또는 null\n- \"mgmt_fee\": number 또는 null\n- \"parking_per_household\": number 또는 null\n- \"elevator\": boolean 또는 null\n- \"mgmt_office\": string 또는 null\n- \"station_name\": string 또는 null\n- \"station_walk_min\": integer 또는 null\n- \"elementary\": string 또는 null\n- \"chopuma\": boolean 또는 null\n- \"middle_high\": string 또는 null\n- \"commercial\": string 또는 null\n- \"defects\": 문자열 목록 (없으면 [])\n- \"uncertain\": 문자열 목록 (없으면 [])"
TEXTS = {
 "scripts": [
  {
   "id": "s1",
   "text": "오늘 첫 번째 매물. 서울 영등포구 당산동 아파트, 24평형. 매매는 7억 2천 부르고 전세는 4억 5천 정도 나온대. 2008년식이고 단지는 650세대. 당산역까지 걸어서 8분. 주차는 세대당 1.2대고 엘리베이터 있어. 관리비는 평균 18만원. 베란다 쪽에 누수 흔적이 좀 있어서 확인 필요하고. 남향이라 채광 괜찮고, 초등학교가 단지 바로 옆이라 초품아야. 융자는 없대."
  },
  {
   "id": "s2",
   "text": "두 번째는 인천 부평구 부평동 아파트. 32평형이고 매매 5억 3천, 전세는 3억 8천까지 받을 수 있대. 2015년식, 1200세대 대단지. 부평역까지 도보 12분이라 역세권은 좀 애매해. 주차는 세대당 1.4대, 지하주차장 연결돼 있고. 관리비는 25만원 정도. 초등학교는 길 하나 건너야 하고 중학교는 걸어서 10분. 단지 앞에 상가랑 학원가 있어. 관리사무소는 깔끔하게 잘 돌아가는 느낌. 15층 중에 9층, 동향."
  },
  {
   "id": "s3",
   "text": "세 번째는 부천 원미구 중동 빌라. 18평. 매매 2억 8천에 전세 2억 3천. 음... 1998년식이라 좀 오래됐어. 엘리베이터 없고 4층인데 계단으로 올라가야 돼. 24세대짜리고 주차는 세대당 0.7대라 빡빡해. 화장실 타일 깨진 데 있고 창틀 쪽에 곰팡이 보였어. 수리비 꽤 들 듯. 대신 역에서 걸어서 5분이라 위치는 좋아. 관리비는 7만원."
  },
  {
   "id": "s4",
   "text": "네 번째 본 거는, 그 뭐지, 인천 연수구 송도동 아파트인데 평수는 한 27평쯤 되는 것 같고. 가격은 중개사가 확실히 말 안 해주고 협의 가능하다고만 했어. 전세는 4억 정도라던데. 연식은 잘 모르겠고 한 10년 좀 넘어 보이던데. 일단 다시 물어봐야 될 것 같아."
  },
  {
   "id": "s5",
   "text": "마지막 매물, 성남 분당구 정자동. 매매가 9억 9천에 전세 3억 8천. 아, 아니다. 전세는 3억 6천이라고 했던 것 같기도 하고. 평수는 23평. 1995년식 구축이고 세대수는 800세대 정도. 관리비 15만원. 주차는 세대당 0.8대라 좀 빡빡할 듯. 정자역까지 걸어서 6분. 엘리베이터 있고. 융자가 1억 정도 잡혀 있다고 해서 이건 좀 봐야 돼."
  }
 ],
 "turbo_noisy_prompt": [
  {
   "id": "s1",
   "text": "오늘 첫 번째 매물. 서울 영등포구 당산동 아파트, 24평형. 매매는 7억 2천 부르고 전세는 4억 5천 정도 나온대. 2008년식이고 단지는 650세대. 당산역까지 걸어서 8분. 주차는 세대당 1.2대구 엘리베이터 있어. 관리비는 평균 18만원. 베란다 쪽에 누수 흔적이 좀 있어서 확인 필요하고. 남향이라 채광 괜찮고 초등학교가 단지 바로 옆이라 초품아야. 융자는 없대."
  },
  {
   "id": "s2",
   "text": "두번째는 인천 부평구 부평동 아파트. 32평형이고 매매 5억 3천, 전세는 3억 8천까지 받을 수 있대. 2015년식 1200세대 대단지. 부평역까지 도보 12분이라 역세권은 좀 애매해. 주차는 세대당 1.4대, 지하주차장 연결돼 있고 관리비는 25만원 정도. 초등학교는 길 하나 건너야 하고 중학교는 걸어서 10분. 단지 앞에 상가랑 학원가 있어. 관리사무소는 깔끔하게 잘 돌아가는 느낌. 15층 중에 9층, 동양."
  },
  {
   "id": "s3",
   "text": "세번째는 부천 원리부 중동 빌라. 18평, 매매 2억 8천에 전세 2억 3천. 음, 1998년식이라 좀 오래됐어. 엘리베이터 없고 4층인데 계단으로 올라가야 돼, 24세대짜리고 주차는 세대당 0.7대라 빡빡해. 화장실 타일 깨진 데 있고 창틀 쪽에 곰팡이 보였어. 수리비 꽤 들듯 대신 역에서 걸어서 5분이라 위치는 좋아. 관리비는 7만원. 관리비는 8천원."
  },
  {
   "id": "s4",
   "text": "네 번째 본 거는 그 뭐지? 인천 연수구 송도동 아파트인데 평수는 한 스물일구평쯤 되는 것 같고. 가격은 중개사가 확실히 말 안해주고 협회 가능하다고만 했어. 전세는 4억 정도라던데. 연식은 잘 모르겠고 한 10년 좀 넘어 보이던데. 일단 다시 물어봐야 될 것 같아."
  },
  {
   "id": "s5",
   "text": "마지막 매물, 성남 분당구 정자동. 매매가 9억 9천에 전세 3억 8천. 아, 아니다. 전세는 3억 6천이라고 했던 것 같기도 하고. 평수는 23평. 1995년식 구축이고 세대수는 800세대 정도. 관리비 15만원. 주차는 세대당 0.8대라 좀 빡빡할 듯 정자역까지 걸어서 6분. 엘리베이터 있고. 융자가 1억 정도 잡혀있다고 해서 이건 좀 봐야 돼."
  }
 ],
 "small_noisy_noprompt": [
  {
   "id": "s1",
   "text": "오늘 첫 번째 메물 서울 영등포구 방산동 아파트 24평형 메매는 7억 2000부르고 전세는 4억 5000정도 나은데 2008년 시기고 단지는 650세대. 방산력까지 걸어서 8분. 주차는 세대당 1.2대고 엘리베이터 있어. 관리비는 평균 18만원. 베란다 쪽에 누수 흔적이 좀 있어서 확인 필요하고 남현이라 체광 괜찮고 초등학교가 단지 바로 옆이라 초프마야. 융자는 어때?"
  },
  {
   "id": "s2",
   "text": "두 번째는 인천 부평구 부평동 아파트 32평형이고 매매 5억 3천 전세는 3억 8천까지 받을 수 있대 2015년식, 1200세대 대단지 부평역까지 도보 12분이라 역세권은 좀 애매해 주차는 세대당 1.4대, 지하주차장 연결돼 있고 관리비는 25만원 정도 초등학교는 길 하나 건너야 하고 중학교는 걸어서 10분 단지 앞에 상가랑 학원가 있어 관리 사무소는 깔끔하게 잘 돌아가는 느낌 15층 중에 9층, 동양"
  },
  {
   "id": "s3",
   "text": "세 번째는 부천원 리부 중동 빌라. 18평, 맴매 2억 8천에 전세 2억 3천. 음... 1998년식이라 좀 오래됐어. 엘리베이터 없고 4층인데 계단으로 올라가야 돼. 24세대짜리고 주차는 세대당 0.7대라 빡빡해. 화장실 타일 깨진데 있고 창틀 쪽에 공팡이 보였어. 수리비 꽤 들듯 대신 역에서 걸어서 5분이라 위치는 좋아? 관리비는 7만원."
  },
  {
   "id": "s4",
   "text": "네 번째 본 거는 그 뭐지? 인천연수구 송도동 아파트인데 평수는 1.27고 평쯤 되는 것 같고 가격은 중계사가 확실히 말 안 해주고 협회 가능하다고만 했어. 전세는 4억 정도라던데. 연식은 잘 모르겠고 한 10년 좀 넘어 보이던데. 일단 다시 물어봐야 될 것 같아."
  },
  {
   "id": "s5",
   "text": "마지막 메물, 선남분당구 정자동. 메매가 9억 9천에 전세 3억 8천. 아, 아니다. 전세는 3억 6천이라고 했던 것 같기도 하고. 평수는 23평. 1995년 시 구축이고, 세대수는 800세대 정도. 관리비 15만원. 주차는 세대당 0.8대라 좀 빡빡할 듯 정자역까지 걸어서 6분. 엘리베이터 있고. 용자가 1억 정도 잡혀 있다고 해서 이건 좀 봐야 돼."
  }
 ]
}

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"))
print("로드 완료")

In [ ]:
import json, re, time

def extract(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "받아쓴 임장 메모:\n\n" + text}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=700, do_sample=False)   # 매번 같은 결과(재현성)
    raw = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{[\s\S]*\}", raw)
    try:
        return json.loads(m.group(0)), raw
    except Exception:
        return None, raw

RESULTS = {}
for cond, items in TEXTS.items():
    RESULTS[cond] = {}
    for it in items:
        t0 = time.time()
        data, raw = extract(it["text"])
        sec = round(time.time() - t0, 1)
        RESULTS[cond][it["id"]] = {"data": data, "seconds": sec, "raw": None if data else raw}
        print(f"{cond:<22} {it['id']}  {sec:>5}초  {'OK' if data else 'JSON 실패'}")

## 결과 출력 (Claude Code가 이 출력을 읽어 저장소에 넣는다)

In [ ]:
print("<<<LLM_JSON>>>" + json.dumps({"model": MODEL, "results": RESULTS}, ensure_ascii=False) + "<<<END>>>")